# Objectif, problématique et démarche

*Cette section présente le contexte du projet, la problématique posée et la démarche suivie pour mettre en production le modèle de scoring crédit.*

**Dépôt GitHub du projet :** [github.com/ouse4444/OC_DS_P8](https://github.com/ouse4444/OC_DS_P8)

## Contexte

La société Prêt à Dépenser propose des crédits à la consommation à des personnes ayant peu ou pas d'historique de prêt. Lors du Projet 6, un modèle de scoring a été développé pour estimer la probabilité de défaut de paiement d'un client, à partir des données Home Credit Default Risk.

Ce modèle a été suivi et versionné avec MLflow. Il s'agit d'un LightGBM optimisé avec un score F-bêta (bêta = 3,16), qui pénalise davantage les défauts non détectés que les bons clients refusés, avec un seuil de décision de 0,36.

Le département Crédit Express souhaite désormais utiliser ce modèle pour traiter les nouvelles demandes en quasi temps réel. La Lead Data Scientist demande une API déployable avec Docker, ainsi qu'un suivi du modèle une fois en production.

## Problématique

Un modèle performant sur un jeu de test n'est pas encore un modèle utilisable. Le Projet 6 a répondu à la question : *le modèle prédit-il bien ?* Le Projet 8 répond à une autre question : *le modèle fonctionne-t-il de façon fiable en production, et dans la durée ?*

Cette question se décline en quatre enjeux :

- **Servir le modèle** : le rendre accessible par une API, avec des entrées validées et des erreurs gérées.
- **Automatiser** : tester, construire et déployer chaque nouvelle version sans intervention manuelle.
- **Surveiller** : collecter les données de production pour détecter une dérive des données ou une anomalie de fonctionnement.
- **Optimiser** : mesurer et réduire le temps de réponse, sans dégrader la qualité des prédictions.

## Démarche

Le projet suit une approche itérative : chaque brique est construite, testée et versionnée avant de passer à la suivante.

| Étape | Objet | Outils |
|---|---|---|
| 1 | Versionner le projet et reprendre le modèle du Projet 6 | Git, GitHub, MLflow |
| 2 | Exposer le modèle par une API, la conteneuriser et automatiser son déploiement | FastAPI, Gradio, Docker, GitHub Actions, Azure |
| 3 | Stocker les données de production et analyser la dérive | PostgreSQL, Evidently, Streamlit |
| 4 | Analyser et optimiser les performances | cProfile, ONNX Runtime |

Ce notebook couvre l'étape 1 : la reprise du modèle et la vérification qu'il reproduit à l'identique les résultats validés au Projet 6. Cette vérification est un préalable à tout déploiement : elle garantit que le modèle servi en production est bien celui qui a été évalué.

# Mise en place de l'environnement Python

*Cette section décrit la création de l'environnement de travail, importe les bibliothèques nécessaires et définit les chemins et paramètres du projet.*

## Création de l'environnement virtuel avec uv

L'environnement virtuel est créé avec **uv** sous **Python 3.12**, la version utilisée au Projet 6. Les commandes suivantes ont été exécutées dans le terminal, à la racine du projet :

```bash
uv venv --python 3.12
source .venv/bin/activate
```

Un environnement virtuel isole les bibliothèques du projet de celles du système et des autres projets. Chaque projet dispose ainsi de ses propres versions, sans conflit.

## Installation des bibliothèques

Les bibliothèques liées au modèle sont installées dans **exactement les mêmes versions qu'au Projet 6**, relevées au préalable dans l'environnement de ce projet :

```bash
uv pip install "lightgbm==4.6.0" "scikit-learn==1.8.0" "mlflow==3.12.0" "pandas==2.3.3" "numpy==2.4.6" ipykernel jupyter
```

Ce choix est essentiel : un modèle sérialisé avec une version donnée de scikit-learn ou de LightGBM peut se recharger avec des erreurs, ou produire des prédictions légèrement différentes, avec une autre version. Aligner les versions garantit que le modèle servi est strictement celui qui a été évalué.

Les bibliothèques propres aux étapes suivantes (FastAPI, Gradio, Evidently, Streamlit) seront ajoutées au fur et à mesure, chacune avec la brique qui l'utilise.

## Enregistrement du kernel Jupyter

L'environnement est déclaré comme kernel Jupyter, pour être sélectionné dans VS Code :

```bash
python -m ipykernel install --user --name oc_ds_p8 --display-name "Python (OC_DS_P8)"
```

Le contrôle des versions réalisé à la cellule suivante confirme que le notebook s'exécute bien dans cet environnement.

## Importation des bibliothèques

In [1]:
# Bibliothèques standard
import sys
from pathlib import Path

# Manipulation des données
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt

# Modèle, suivi MLflow et évaluation
import lightgbm
import mlflow
import mlflow.sklearn
import sklearn
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

# Contrôle des versions : elles doivent être identiques à celles du Projet 6
print(f"Interpreter  : {sys.executable}")
print(f"Python       : {sys.version.split()[0]}")
print(f"lightgbm     : {lightgbm.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"mlflow       : {mlflow.__version__}")
print(f"pandas       : {pd.__version__}")
print(f"numpy        : {np.__version__}")

Interpreter  : /Users/ousmane/Desktop/OPENCLASSROOMS/Data Scientist Machine learning/OC_DS_P8/.venv/bin/python
Python       : 3.12.8
lightgbm     : 4.6.0
scikit-learn : 1.8.0
mlflow       : 3.12.0
pandas       : 2.3.3
numpy        : 2.4.6


## Chemins et paramètres du projet

Les chemins sont construits à partir de la racine du projet, pour que le notebook fonctionne quel que soit le poste de travail. Les paramètres du Projet 6 (nom de la cible, graine aléatoire) sont lus directement dans son fichier de configuration, plutôt que recopiés à la main : une seule source de vérité évite toute divergence.

In [2]:
# Racine du projet : le notebook est exécuté depuis le dossier notebooks/
PROJECT_ROOT = Path("..").resolve()

# Emplacements des artefacts utilisés dans ce notebook
MODEL_DIR = PROJECT_ROOT / "model"
REFERENCE_DATA = PROJECT_ROOT / "data" / "processed" / "application_train_enriched.parquet"
P6_MLRUNS = PROJECT_ROOT.parent / "OC_DS_P6" / "credit-scoring-mlops" / "mlruns"

# Paramètres du Projet 6, lus dans sa configuration pour garantir la cohérence
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from credit_scoring.config import TARGET, RANDOM_STATE

# Seuil de décision retenu au Projet 6 par optimisation du F-bêta
THRESHOLD = 0.36

print(f"Project root  : {PROJECT_ROOT}")
print(f"Target        : {TARGET}")
print(f"Random state  : {RANDOM_STATE}")
print(f"Threshold     : {THRESHOLD}")

Project root  : /Users/ousmane/Desktop/OPENCLASSROOMS/Data Scientist Machine learning/OC_DS_P8
Target        : TARGET
Random state  : 42
Threshold     : 0.36


## Versionnement du projet

Le projet est versionné avec Git et publié sur un dépôt GitHub public : [github.com/ouse4444/OC_DS_P8](https://github.com/ouse4444/OC_DS_P8).

- **Fichier `.gitignore`** : il exclut l'environnement virtuel, les secrets (fichier `.env`), le suivi MLflow local et les **données**. Les données Home Credit sont assimilées à des données clients : elles ne doivent jamais être publiées. Seul le script qui les reconstruit est versionné.
- **Stratégie de branches** : la branche `main` porte les versions stables, destinées au déploiement. La branche `develop` sert au travail courant. Chaque brique est développée sur une branche dédiée, puis fusionnée par *pull request*.
- **Commits explicites** : chaque étape fait l'objet d'un commit, avec un préfixe qui en indique la nature (`chore` pour l'organisation, `feat` pour une fonctionnalité, `docs` pour la documentation).

L'historique des commits à ce stade :

In [3]:
# Historique des commits du dépôt, toutes branches confondues
# L'option --no-color supprime les codes de couleur du terminal, illisibles dans un notebook
!git -C .. --no-pager log --oneline --all --no-color


8a8b4e4 (HEAD -> feature/notebook-demarrage, origin/feature/notebook-demarrage) docs: add start-up notebook (model reuse and reproducibility check)
a5a3d98 (origin/develop, develop) feat: export validated P6 model (LightGBM F-beta, threshold 0.36)
9fa93ad feat: add script to rebuild the P6 enriched reference dataset
81dc455 docs: add project 6 F-beta notebook as model reference
777a9fa feat: import credit_scoring source code from project 6
a290621 (origin/main, main) chore: initialise project structure and .gitignore


In [4]:
# Export depuis le registre MLflow du Projet 6, en lecture seule
if not (MODEL_DIR / "model.pkl").exists():
    mlflow.set_tracking_uri(str(P6_MLRUNS))
    mlflow.artifacts.download_artifacts(
        artifact_uri="models:/credit_scoring_lightgbm_fbeta/1",
        dst_path=str(MODEL_DIR),
    )
    print("Model exported from the MLflow registry.")
else:
    print("Model already exported: no new download.")

# Contenu du dossier exporté
for f in sorted(MODEL_DIR.iterdir()):
    print(f"{f.name:<25} {f.stat().st_size / 1024:>8.1f} KB")

Model already exported: no new download.
MLmodel                        0.8 KB
conda.yaml                     0.3 KB
model.pkl                    574.4 KB
python_env.yaml                0.1 KB
registered_model_meta          0.1 KB
requirements.txt               0.2 KB


## Structure du modèle

Le modèle est rechargé avec MLflow, puis sa structure est examinée. Cette étape détermine ce que l'API devra transmettre au modèle : des données brutes ou des données déjà préparées.

In [5]:
# Rechargement du modèle exporté, sans passer par le registre
model = mlflow.sklearn.load_model(str(MODEL_DIR))

# Étapes de premier niveau du pipeline
print(f"Model type: {type(model).__name__}\n")
for name, step in model.steps:
    print(f"Step '{name}': {type(step).__name__}")

# Détail de l'étape de préparation
print("\nPreparation sub-steps:")
for name, step in model.named_steps["preparation"].steps:
    print(f"  - '{name}': {type(step).__name__}")

# Nombre de variables reçues par le LightGBM après préparation
print(f"\nFeatures seen by LightGBM: {model.named_steps['modele'].n_features_in_}")

Model type: Pipeline

Step 'preparation': Pipeline
Step 'modele': LGBMClassifier

Preparation sub-steps:
  - 'features': ConstructeurFeatures
  - 'preparation': ColumnTransformer

Features seen by LightGBM: 314


**Lecture du résultat :**

Le modèle n'est pas un LightGBM seul, mais un **pipeline complet** en deux étapes :

1. **La préparation** : la construction des variables métier (taux d'effort, endettement, âge, ancienneté), l'imputation des valeurs manquantes, l'encodage des variables catégorielles et la standardisation.
2. **Le modèle LightGBM**, qui reçoit 314 variables après préparation.

**Conséquence pour l'API :** la préparation des données est incluse dans le modèle. L'API transmettra les données du client telles quelles, et le pipeline appliquera exactement les mêmes transformations qu'à l'entraînement. Ce choix évite un risque classique en production : recoder la préparation à part et introduire un écart entre l'entraînement et l'inférence.

Le modèle est également autonome : la classe de préparation des variables, définie dans le notebook du Projet 6, a été enregistrée par MLflow à l'intérieur du fichier du modèle.


# Étape 2 : Reconstruction des données de référence

*Cette section reconstruit le jeu de données sur lequel le modèle a été entraîné. Il servira de référence pour l'analyse de la dérive et de source de clients de test pour l'API.*

## Pourquoi une reconstruction

Le fichier de données sauvegardé au Projet 6 (`application_train_fe.parquet`, 130 colonnes) ne contenait pas les agrégats des tables secondaires. Ces agrégats avaient été calculés dans le notebook du Projet 6, en mémoire, puis fusionnés à la table des clients sans être enregistrés sur le disque. Un premier test de prédiction sur ce fichier a d'ailleurs échoué, faute de ces colonnes.

Un script dédié, `scripts/build_reference_data.py`, reproduit donc à l'identique les cellules du notebook du Projet 6, à partir de l'archive Kaggle d'origine :

| Étape | Résultat |
|---|---|
| Table des clients réduite, nombre de documents fournis, anomalie `DAYS_EMPLOYED` | 65 colonnes |
| Agrégation en cascade de `bureau_balance` puis `bureau` | 58 colonnes `BUREAU_` |
| Agrégation de `previous_application` | 30 colonnes `PREV_` |
| Agrégation de `POS_CASH_balance` | 17 colonnes `POS_` |
| Agrégation de `installments_payments` | 12 colonnes `INSTAL_` |
| Agrégation de `credit_card_balance` | 19 colonnes `CC_` |
| Fusion par jointure à gauche sur l'identifiant client | 307 511 lignes, 201 colonnes |

Le choix d'un script plutôt que de cellules de notebook est délibéré : la reconstruction doit pouvoir être relancée en une seule commande, par exemple dans le pipeline de monitoring.

Les données ne sont pas versionnées dans Git, car elles sont assimilées à des données clients. Le script, lui, est versionné : il permet de reconstruire les données à l'identique.

In [6]:
import subprocess

# Archive Kaggle d'origine, conservée dans le Projet 6 pour éviter de dupliquer 688 Mo
RAW_ZIP = PROJECT_ROOT.parent / "OC_DS_P6" / "credit-scoring-mlops" / "data" / "raw" / "home-credit-default-risk.zip"

# La reconstruction n'est lancée que si le fichier n'existe pas encore,
# car elle prend plusieurs minutes (27 millions de lignes dans bureau_balance)
if not REFERENCE_DATA.exists():
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "build_reference_data.py"),
         "--zip-path", str(RAW_ZIP), "--output", str(REFERENCE_DATA)],
        check=True,
    )
else:
    print("Reference dataset already built: loading existing file.")

df = pd.read_parquet(REFERENCE_DATA)
print(f"Shape        : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Default rate : {df[TARGET].mean() * 100:.2f} %")

Reference dataset already built: loading existing file.
Shape        : 307,511 rows x 201 columns
Default rate : 8.07 %


In [7]:
# Répartition des variables explicatives par table d'origine
# L'identifiant et la cible ne sont pas des variables explicatives
features = [c for c in df.columns if c not in (TARGET, "SK_ID_CURR")]

sources = {
    "Credit bureau (BUREAU_)": "BUREAU_",
    "Previous applications (PREV_)": "PREV_",
    "POS cash balance (POS_)": "POS_",
    "Installments (INSTAL_)": "INSTAL_",
    "Credit card (CC_)": "CC_",
}
counts = {label: sum(c.startswith(p) for c in features) for label, p in sources.items()}
counts = {"Client application": len(features) - sum(counts.values()), **counts}

summary = pd.DataFrame.from_dict(counts, orient="index", columns=["Number of features"])
summary.loc["Total"] = summary["Number of features"].sum()
summary

,Number of features
Client application,63
Credit bureau (BUREAU_),58
Previous applications (PREV_),30
POS cash balance (POS_),17
Installments (INSTAL_),12
Credit card (CC_),19
Total,199


**Lecture du résultat :** le jeu reconstruit compte 307 511 clients et 199 variables explicatives, avec un taux de défaut de 8,07 %, identique au Projet 6. Ces 199 variables constituent l'entrée attendue par le modèle, et donc par l'API.

# Étape 3 : Vérification de la reproductibilité

*Cette section vérifie que le modèle exporté, appliqué aux données reconstruites, reproduit exactement les résultats validés au Projet 6.*

## Méthode

Le découpage entraînement/test du Projet 6 est reproduit à l'identique : 80 % / 20 %, stratifié sur la cible, avec la même graine aléatoire. Le modèle est ensuite appliqué au jeu de test, au seuil de décision de 0,36.

Si les données et le modèle sont strictement identiques, la matrice de confusion doit être identique au chiffre près. C'est un test plus exigeant qu'une comparaison de métriques arrondies.

In [8]:
import warnings

# La préparation transmet au LightGBM un tableau sans noms de colonnes, alors qu'il a été
# entraîné avec des noms : scikit-learn émet un avertissement. L'ordre des colonnes est
# néanmoins garanti par le pipeline, d'où le masquage de ce seul avertissement.
warnings.filterwarnings("ignore", message="X does not have valid feature names")

# Découpage identique au Projet 6
X = df.drop(columns=[TARGET, "SK_ID_CURR"])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

# Probabilités de défaut, puis décision au seuil métier
proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= THRESHOLD).astype(int)

print(f"Test set: {len(X_test):,} clients")

Test set: 61,503 clients


In [9]:
# Comparaison avec les résultats validés au Projet 6
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

# Le type « object » conserve chaque valeur dans son format d'origine :
# les effectifs restent des entiers, le rappel et la précision des décimaux
comparison = pd.DataFrame(
    {
        "Project 6": [30134, 26404, 735, 4230, 0.852, 0.138],
        "Project 8": [int(tn), int(fp), int(fn), int(tp),
                      round(recall_score(y_test, pred), 3),
                      round(precision_score(y_test, pred), 3)],
    },
    index=["True negatives", "False positives", "False negatives",
           "True positives", "Recall", "Precision"],
    dtype=object,
)
comparison["Identical"] = comparison["Project 6"] == comparison["Project 8"]

print(f"AUC (new reference for optimisation): {roc_auc_score(y_test, proba):.4f}")
comparison

AUC (new reference for optimisation): 0.7855


,Project 6,Project 8,Identical
True negatives,30134,30134,True
False positives,26404,26404,True
False negatives,735,735,True
True positives,4230,4230,True
Recall,0.852,0.852,True
Precision,0.138,0.138,True


# Conclusion et suite du projet

## Bilan de l'étape 1

- Le projet est versionné sur un dépôt GitHub public, avec une stratégie de branches et des commits explicites.
- L'environnement est aligné sur celui du Projet 6, ce qui garantit un rechargement fidèle du modèle.
- Le modèle a été exporté depuis le registre MLflow. Il s'agit d'un pipeline autonome qui inclut la préparation des données.
- Les données d'entraînement ont été reconstruites par un script reproductible.
- Le modèle exporté reproduit exactement les résultats du Projet 6 : il peut être mis en production.

## Choix techniques pour la suite

| Brique | Choix | Justification |
|---|---|---|
| API | FastAPI | Validation des entrées et des sorties avec Pydantic, documentation Swagger automatique, chargement unique du modèle au démarrage |
| Interface de test | Gradio | Démonstration interactive de l'API, intégrée à la même application |
| Conteneurisation | Docker | Environnement identique en local, dans la CI et en production |
| CI/CD | GitHub Actions | Tests, construction de l'image et déploiement à chaque push |
| Hébergement | Azure Container Apps | Déploiement réel dans le cloud, avec un forfait gratuit mensuel ; outil utilisé dans un contexte professionnel |
| Stockage de production | PostgreSQL | Données structurées, faciles à interroger pour l'analyse |
| Dérive des données | Evidently | Tests statistiques et rapports visuels dédiés à la dérive |
| Tableau de bord | Streamlit | Suivi interactif des scores, de la latence, des erreurs et de la dérive |

## Prochaine étape

L'étape suivante consiste à développer l'API en local : chargement du modèle au démarrage, route de prédiction, validation des données et gestion des erreurs, puis interface Gradio.